In [ ]:
"""Extract Urban Plumber station UCP data into one CSV file."""

In [ ]:
import math
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import xarray as xr

In [ ]:
GLOBUCP_ROOT = Path("/tera12/yuanhua/dongwz/urban_data/raw_urban/GlobUCP/all_tiles")
GLAMOUR_BF_ROOT = Path("/tera12/yuanhua/dongwz/urban_data/raw_urban/GLAMOUR/BF")
GLAMOUR_BH_ROOT = Path("/tera12/yuanhua/dongwz/urban_data/raw_urban/GLAMOUR/BH")
SITE_ROOT = Path("/tera12/yuanhua/data/CoLMpointdata/Urban-PLUMBER2/Sitedata")
NCAR_PHYS_PATH = Path("/tera12/yuanhua/data/CoLMrawdata/urban_physical/Urban_Physical_Parameter.NCAR.nc")

In [ ]:
GLOBUCP_DX = 0.008333333333
GLOBUCP_DY = 0.008333333333
GLOBUCP_KNOWN_LON = -179.995834
GLOBUCP_KNOWN_LAT = -89.995834
GLOBUCP_TILE_X = 120
GLOBUCP_TILE_Y = 120
GLOBUCP_TILE_Z = 132
GLOBUCP_SCALE = 0.0001
GLOBUCP_FRACTION_INDEX = 90
GLOBUCP_HEIGHT_INDEX = 91

In [ ]:
wtroof_lcz = [0.5, 0.5, 0.55, 0.3, 0.3, 0.3, 0.8, 0.4, 0.15, 0.25]
htroof_lcz = [45.0, 15.0, 5.0, 40.0, 15.0, 5.0, 3.0, 7.0, 5.0, 8.5]

In [ ]:
ds_ncar = None

In [ ]:
def read_site_info(path):
    path = Path(path)
    info = pd.read_csv(path)

    for col in ("Lat", "Lon", "Year"):
        if col in info.columns:
            info[col] = pd.to_numeric(info[col], errors="coerce")
    return info

In [ ]:
def find_15deg_folder(lon, lat):
    lon0 = math.floor(lon / 15.0) * 15
    lat0 = math.floor(lat / 15.0) * 15
    lon1 = lon0 + 14
    lat1 = lat0 + 14
    return f"GloUCP-X{lon0}_{lon1}.Y{lat0}_{lat1}"

In [ ]:
def lonlat_to_global_index(lon, lat):
    x_global = int(math.floor((lon - GLOBUCP_KNOWN_LON) / GLOBUCP_DX)) + 1
    y_global = int(math.floor((lat - GLOBUCP_KNOWN_LAT) / GLOBUCP_DY)) + 1
    return x_global, y_global

In [ ]:
def global_index_to_1deg_filename(x_global, y_global):
    x_start = ((x_global - 1) // GLOBUCP_TILE_X) * GLOBUCP_TILE_X + 1
    x_end = x_start + GLOBUCP_TILE_X - 1
    y_start = ((y_global - 1) // GLOBUCP_TILE_Y) * GLOBUCP_TILE_Y + 1
    y_end = y_start + GLOBUCP_TILE_Y - 1
    fname = f"{x_start:05d}-{x_end:05d}.{y_start:05d}-{y_end:05d}"
    return fname, x_start, y_start

In [ ]:
def read_gloucp_file(path):
    raw = np.fromfile(path, dtype=">u4")
    expected = GLOBUCP_TILE_Z * GLOBUCP_TILE_Y * GLOBUCP_TILE_X
    if raw.size != expected:
        raise ValueError(f"Unexpected file size for {path}: {raw.size} != {expected}")
    return raw.reshape((GLOBUCP_TILE_Z, GLOBUCP_TILE_Y, GLOBUCP_TILE_X)).astype(np.float64) * GLOBUCP_SCALE

In [ ]:
def extract_gloucp_station_values(lon, lat):
    folder = find_15deg_folder(lon, lat)
    x_global, y_global = lonlat_to_global_index(lon, lat)
    fname, x_start, y_start = global_index_to_1deg_filename(x_global, y_global)
    file_path = GLOBUCP_ROOT / folder / fname
    if not file_path.exists():
        return np.nan, np.nan

    data = read_gloucp_file(file_path)
    ix = x_global - x_start
    iy = y_global - y_start
    values = data[:, iy, ix]
    return values[GLOBUCP_HEIGHT_INDEX], values[GLOBUCP_FRACTION_INDEX]

In [ ]:
def glamour_tile_path(root, prefix, lon, lat):
    lon_start = math.floor(lon / 9.0) * 9
    lat_start = math.floor(lat / 9.0) * 9
    lon_end = lon_start + 9
    lat_end = lat_start + 9
    return root / f"{prefix}_{lon_start}_{lon_end}_{lat_start}_{lat_end}.tif"

In [ ]:
def extract_glamour_value(root, prefix, lon, lat):
    tile_path = glamour_tile_path(root, prefix, lon, lat)
    if not tile_path.exists():
        return np.nan

    with rasterio.open(tile_path) as ds:
        row, col = ds.index(lon, lat)
        if row < 0 or row >= ds.height or col < 0 or col >= ds.width:
            return np.nan
        value = ds.read(1)[row, col]
        nodata = ds.nodata
        if nodata is not None and np.isclose(value, nodata):
            return np.nan
        return float(value)

In [ ]:
def extract_glamour_station_values(lon, lat):
    height = extract_glamour_value(GLAMOUR_BH_ROOT, "BH", lon, lat)
    fraction = extract_glamour_value(GLAMOUR_BF_ROOT, "BF", lon, lat)
    return height, fraction

In [ ]:
def extract_nc_value(ds, var_name, lat, lon):
    i_lat = int(np.argmin(np.abs(ds.lat.values - lat)))
    i_lon = int(np.argmin(np.abs(ds.lon.values - lon)))
    return float(np.asarray(ds[var_name].isel(lat=i_lat, lon=i_lon).values).squeeze())

In [ ]:
def scalar_value(value):
    data = np.asarray(value, dtype=float).squeeze()
    if data.ndim == 0:
        return float(data)

    finite = data[np.isfinite(data)]
    if finite.size == 0:
        return np.nan
    return float(finite.flat[0])

In [ ]:
def extract_site_value(ds, var_name):
    if var_name not in ds:
        return np.nan
    return scalar_value(ds[var_name].values)

In [ ]:
def id_to_index(value, size):
    if pd.isna(value):
        return None

    index = int(value) - 1
    if index < 0 or index >= size:
        return None
    return index

In [ ]:
def extract_obs_station_values(info):
    obs_wt, obs_ht = [], []

    for site in info["site"]:
        site_file = SITE_ROOT / f"{site}_site_v1.nc"
        if not site_file.exists():
            obs_wt.append(np.nan)
            obs_ht.append(np.nan)
            continue

        with xr.open_dataset(site_file) as ds:
            obs_wt.append(extract_site_value(ds, "roof_area_fraction"))
            obs_ht.append(extract_site_value(ds, "building_mean_height"))

    return {
        "obs_fraction": obs_wt,
        "obs_height": obs_ht,
    }

In [ ]:
def get_ncar_physical_dataset():
    global ds_ncar
    if ds_ncar is None and NCAR_PHYS_PATH.exists():
        ds_ncar = xr.open_dataset(NCAR_PHYS_PATH)
    return ds_ncar

In [ ]:
def extract_ncar_lut_value(ds, var_name, region_id, urban_class):
    if ds is None or var_name not in ds or pd.isna(region_id) or pd.isna(urban_class):
        return np.nan

    region_dim = ds[var_name].dims[0]
    urban_dim = ds[var_name].dims[1]
    region_index = id_to_index(region_id, ds.sizes[region_dim])
    urban_index = id_to_index(urban_class, ds.sizes[urban_dim])
    if region_index is None or urban_index is None:
        return np.nan

    return scalar_value(ds[var_name].isel(
        {region_dim: region_index, urban_dim: urban_index}
    ).values)

In [ ]:
def extract_external_products(info):
    ghsl_ht, li_ht, ghsl_wt, li_wt = [], [], [], []
    ncar_ht, ncar_wt = [], []
    lcz_ht, lcz_wt = [], []
    gloucp_ht, gloucp_wt = [], []
    glamour_ht, glamour_wt = [], []

    for lat, lon, iyear in zip(info["Lat"], info["Lon"], info["Year"]):
        if pd.isna(lat) or pd.isna(lon):
            ghsl_ht.append(np.nan)
            li_ht.append(np.nan)
            ghsl_wt.append(np.nan)
            li_wt.append(np.nan)
            gloucp_ht.append(np.nan)
            gloucp_wt.append(np.nan)
            glamour_ht.append(np.nan)
            glamour_wt.append(np.nan)
            ncar_ht.append(np.nan)
            ncar_wt.append(np.nan)
            lcz_ht.append(np.nan)
            lcz_wt.append(np.nan)
            continue

        reg_slat = (int(lat / 5) * 5) if lat >= 0 else (int(lat / 5) * 5 - 5)
        reg_elat = reg_slat + 5
        reg_slon = (int(lon / 5) * 5) if lon >= 0 else (int(lon / 5) * 5 - 5)
        reg_elon = reg_slon + 5
        base = f"RG_{reg_elat}_{reg_slon}_{reg_slat}_{reg_elon}"

        ghsl_path = None
        if not pd.isna(iyear):
            ghsl_path = Path(
                f"/tera12/yuanhua/data/CoLMrawdata/urban_morphology/roof_height_fraction_GHSL/"
                f"{base}.ROOF500m.GHSL.{int(iyear)}.nc"
            )
        if ghsl_path is not None and ghsl_path.exists():
            with xr.open_dataset(ghsl_path) as ds:
                ghsl_ht.append(extract_nc_value(ds, "HT_ROOF", lat, lon))
                ghsl_wt.append(extract_nc_value(ds, "PCT_ROOF", lat, lon))
        else:
            ghsl_ht.append(np.nan)
            ghsl_wt.append(np.nan)

        li_path = Path(
            f"/tera12/yuanhua/data/CoLMrawdata/urban_morphology/roof_height_fraction_Li/"
            f"{base}.ROOF1km.Li.nc"
        )
        if li_path.exists():
            with xr.open_dataset(li_path) as ds:
                li_ht.append(extract_nc_value(ds, "HT_ROOF", lat, lon))
                li_wt.append(extract_nc_value(ds, "PCT_ROOF", lat, lon))
        else:
            li_ht.append(np.nan)
            li_wt.append(np.nan)

        #ncar_path = Path(
            #f"/tera12/yuanhua/data/CoLMrawdata/urban_type/ncar/"
            #f"{base}.URBTYPE1km_NCAR.Jackson.nc"
        #)
        ncar_path = Path(
            f"/tera12/yuanhua/data/CoLMrawdata/urban_type/"
            f"{base}.URBTYP.nc"
        )
        if ncar_path.exists():
            with xr.open_dataset(ncar_path) as ds:
                urb_class = extract_nc_value(ds, "URBAN_DENSITY_CLASS", lat, lon)
                reg_class = extract_nc_value(ds, "REGION_ID", lat, lon)
                ncar_para = get_ncar_physical_dataset()
                if urb_class == 0:
                    urb_class = 3

                lut_ht = extract_ncar_lut_value(ncar_para, "HT_ROOF", reg_class, urb_class)
                lut_wt = extract_ncar_lut_value(ncar_para, "WTLUNIT_ROOF", reg_class, urb_class)

                ncar_ht.append(lut_ht)
                ncar_wt.append(lut_wt)
        else:
            ncar_ht.append(np.nan)
            ncar_wt.append(np.nan)

        lcz_path = Path(
            f"/tera12/yuanhua/data/CoLMrawdata/urban_type/lcz/"
            f"{base}.URBTYPE500m_LCZ.Demuzere.nc"
        )
        if lcz_path.exists():
            with xr.open_dataset(lcz_path) as ds:
                urb_class = extract_nc_value(ds, "LCZ", lat, lon)
                class_index = id_to_index(urb_class, len(wtroof_lcz))

                if class_index is not None:
                    lut_ht = htroof_lcz[class_index]
                    lut_wt = wtroof_lcz[class_index]
                else:
                    lut_ht = np.nan
                    lut_wt = np.nan

                lcz_ht.append(lut_ht)
                lcz_wt.append(lut_wt)
        else:
            lcz_ht.append(np.nan)
            lcz_wt.append(np.nan)

        glob_height, glob_fraction = extract_gloucp_station_values(lon, lat)
        gloucp_ht.append(glob_height)
        gloucp_wt.append(glob_fraction)

        glamour_height, glamour_fraction = extract_glamour_station_values(lon, lat)
        glamour_ht.append(glamour_height)
        glamour_wt.append(glamour_fraction)

    return {
        "li_built_height": li_ht,
        "li_built_fraction": li_wt,
        "ghsl_height": ghsl_ht,
        "ghsl_fraction": ghsl_wt,
        "globucp_height": gloucp_ht,
        "globucp_fraction": gloucp_wt,
        "glamour_height": glamour_ht,
        "glamour_fraction": glamour_wt,
        "ud_lut_fraction": ncar_wt,
        "ud_lut_height": ncar_ht,
        "lcz_lut_fraction": lcz_wt,
        "lcz_lut_height": lcz_ht,
    }

In [ ]:
def build_ucp_csv(site_info=None, output_csv="UCPs_data.csv"):
    info = read_site_info(site_info)

    output = info[["site"]].copy()
    output = output.assign(**extract_obs_station_values(info))
    output = output.assign(**extract_external_products(info))
    output.to_csv(output_csv, index=False)
    return output

In [ ]:
def main():
    data = build_ucp_csv(site_info='../SiteInfo.csv')
    print(f"Wrote UCPs_data.csv with {len(data)} rows and {len(data.columns)} columns")

In [ ]:
if __name__ == "__main__":
    main()